# LiCSBAS Jakarta rerun — matched velocity and uncertainty stack

**Frame:** `098A_09673_121312` (ascending)  
**Period:** 2020-01-01 to 2023-12-31  
**Pinned COMET-LiCSBAS commit:** `be6a4991182da900b6cdbfece91149028eb02948`

This notebook regenerates `vel`, `vstd`, `coh_avg`, and `n_unw` from one inversion. It stops after step 14; quality masking is performed reproducibly in the risk-zoning pipeline.

Important operating rules:

1. Colab preserves output text from dead sessions. A cell that appears finished but streams no new output did not rerun.
2. Keep the tab active during the download.
3. Run the verified GEOC backup cell immediately after step 01. Do not continue until it confirms a non-empty archive.
4. Never combine a velocity file from one inversion with `vstd` from another.
5. Every external command below checks its return code and required outputs before reporting success.


In [ ]:
# 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2 — Install the pinned COMET fork and dependencies
import os, sys, subprocess, pathlib

LICSBAS_REPO = pathlib.Path('/content/LiCSBAS')
LICSBAS_COMMIT = 'be6a4991182da900b6cdbfece91149028eb02948'
if not LICSBAS_REPO.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/comet-licsar/LiCSBAS.git', str(LICSBAS_REPO)],
        check=True,
    )
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'checkout', '--detach', LICSBAS_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ['git', '-C', str(LICSBAS_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_commit == LICSBAS_COMMIT, (actual_commit, LICSBAS_COMMIT)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'h5py', 'statsmodels', 'astropy', 'cmcrameri', 'bottleneck', 'rasterio'
], check=True)
requirements = LICSBAS_REPO / 'requirements.txt'
if requirements.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('LiCSBAS commit:', actual_commit)


In [ ]:
# 3 — Configuration and command runner
import json, hashlib, datetime, glob

FRAME = '098A_09673_121312'
START = '20200101'
END = '20231231'
NLOOK = 10
WORK = pathlib.Path('/content/insar')
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/InSAR_Risk_Zoning_Project')
ARCHIVE = DRIVE_OUT / 'LiCSBAS_archive_2020_2023'
BIN = LICSBAS_REPO / 'bin'
LIB = LICSBAS_REPO / 'LiCSBAS_lib'
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
ARCHIVE.mkdir(parents=True, exist_ok=True)

run_env = os.environ.copy()
run_env['PYTHONPATH'] = str(LIB)

def resolve_script(role, preferred, fragments):
    exact = BIN / preferred
    if exact.exists():
        return exact
    candidates = sorted(
        p for p in BIN.glob('LiCSBAS*.py')
        if all(fragment.lower() in p.name.lower() for fragment in fragments)
    )
    if len(candidates) != 1:
        raise RuntimeError(f'Cannot resolve {role}: {candidates}')
    return candidates[0]

SCRIPTS = {
    'download': resolve_script('download', 'LiCSBAS01_get_geotiff.py', ('01', 'get_geo')),
    'multilook': resolve_script('multilook', 'LiCSBAS02_ml_prep.py', ('02', 'ml')),
    'check': resolve_script('check', 'LiCSBAS11_check_unw.py', ('11', 'check_unw')),
    'loop': resolve_script('loop', 'LiCSBAS12_loop_closure.py', ('12', 'loop')),
    'invert': resolve_script('invert', 'LiCSBAS13_sb_inv.py', ('13', 'sb_inv')),
    'vstd': resolve_script('vstd', 'LiCSBAS14_vel_std.py', ('14', 'vel_std')),
    'geotiff': resolve_script('GeoTIFF export', 'LiCSBAS_flt2geotiff.py', ('flt2geotiff',)),
}

def run_script(role, args, required=()):
    cmd = [sys.executable, str(SCRIPTS[role]), *map(str, args)]
    print('RUN:', ' '.join(cmd), flush=True)
    completed = subprocess.run(cmd, cwd=WORK, env=run_env)
    if completed.returncode != 0:
        raise RuntimeError(f'{role} failed with return code {completed.returncode}')
    missing = [str(p) for p in required if not pathlib.Path(p).exists()]
    if missing:
        raise RuntimeError(f'{role} completed but required output is missing: {missing}')
    return completed

print(json.dumps({k: str(v) for k, v in SCRIPTS.items()}, indent=2))


In [ ]:
# 4 — Validate every resolved script and inspect step-02 flags
for role, script in SCRIPTS.items():
    help_run = subprocess.run(
        [sys.executable, str(script), '-h'], cwd=WORK, env=run_env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if help_run.returncode not in (0, 1):
        raise RuntimeError(f'{role} help check failed ({help_run.returncode})')
    print(f'--- {role}: {script.name} ---')
    print('\n'.join(help_run.stdout.splitlines()[:12]))

step02_help = subprocess.run(
    [sys.executable, str(SCRIPTS['multilook']), '-h'], cwd=WORK, env=run_env,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
).stdout
for required_flag in ('-i', '-o', '-n'):
    if required_flag not in step02_help:
        raise RuntimeError(f'Step 02 does not advertise required flag {required_flag}')
print('All command/help checks passed.')


## Optional restore after a lost Colab session

Run the next cell only when a verified GEOC archive already exists in Drive. It restores the expensive step-01 download before continuing.


In [ ]:
# 5 — OPTIONAL restore (leave commented unless needed)
GEOC_ARCHIVE = ARCHIVE / 'GEOC_20200101_20231231.tar.gz'
# if not GEOC_ARCHIVE.exists():
#     raise FileNotFoundError(GEOC_ARCHIVE)
# subprocess.run(['tar', '-xzf', str(GEOC_ARCHIVE), '-C', str(WORK)], check=True)
# if not (WORK / 'GEOC').exists():
#     raise RuntimeError('Archive extraction did not produce WORK/GEOC')
# print('Restored:', GEOC_ARCHIVE)


In [ ]:
# 6 — Step 01: download GEOC products (long-running)
run_script(
    'download', ['-f', FRAME, '-s', START, '-e', END],
    required=[WORK / 'GEOC'],
)
if not any((WORK / 'GEOC').iterdir()):
    raise RuntimeError('GEOC exists but is empty')
print('GEOC entries:', sum(1 for _ in (WORK / 'GEOC').iterdir()))


In [ ]:
# 7 — BACK UP GEOC IMMEDIATELY and verify the archive
GEOC_ARCHIVE = ARCHIVE / 'GEOC_20200101_20231231.tar.gz'
backup = subprocess.run(
    ['tar', '-czf', str(GEOC_ARCHIVE), '-C', str(WORK), 'GEOC']
)
if backup.returncode != 0 or not GEOC_ARCHIVE.exists() or GEOC_ARCHIVE.stat().st_size == 0:
    raise RuntimeError('GEOC backup failed; do not continue')
listing = subprocess.run(
    ['tar', '-tzf', str(GEOC_ARCHIVE)], check=True,
    text=True, stdout=subprocess.PIPE,
).stdout.splitlines()
if len(listing) < 2:
    raise RuntimeError('GEOC archive verification found no content')
print(f'VERIFIED backup: {GEOC_ARCHIVE} ({GEOC_ARCHIVE.stat().st_size / 1e9:.2f} GB, {len(listing):,} entries)')


In [ ]:
# 8 — Step 02: multilook x10
run_script(
    'multilook', ['-i', 'GEOC', '-o', 'GEOCml10', '-n', NLOOK],
    required=[WORK / 'GEOCml10'],
)


In [ ]:
# 9 — Steps 11 and 12: quality check and loop closure
run_script('check', ['-d', 'GEOCml10', '-t', 'TS_GEOCml10'], required=[WORK / 'TS_GEOCml10'])
run_script('loop', ['-d', 'GEOCml10', '-t', 'TS_GEOCml10'], required=[WORK / 'TS_GEOCml10' / 'info'])


In [ ]:
# 10 — Steps 13 and 14: SBAS inversion and velocity uncertainty
run_script(
    'invert', ['-d', 'GEOCml10', '-t', 'TS_GEOCml10', '--mem_size', '4000'],
    required=[WORK / 'TS_GEOCml10' / 'cum.h5', WORK / 'TS_GEOCml10' / 'results' / 'vel'],
)
run_script(
    'vstd', ['-t', 'TS_GEOCml10'],
    required=[WORK / 'TS_GEOCml10' / 'results' / 'vstd'],
)


In [ ]:
# 11 — Export the matched four-layer GeoTIFF stack
results_dir = WORK / 'TS_GEOCml10' / 'results'
dem_par = WORK / 'GEOCml10' / 'EQA.dem_par'
layers = ('vel', 'vstd', 'coh_avg', 'n_unw')
exported = {}
for layer in layers:
    source = results_dir / layer
    if not source.exists():
        raise FileNotFoundError(f'Missing LiCSBAS result layer: {source}')
    destination = DRIVE_OUT / f'jakarta_{layer}_2020_2023.tif'
    run_script(
        'geotiff', ['-i', source, '-p', dem_par, '-o', destination],
        required=[destination],
    )
    exported[layer] = destination
print(json.dumps({k: str(v) for k, v in exported.items()}, indent=2))


In [ ]:
# 12 — Validate grids/values and write a provenance manifest
import numpy as np
import rasterio

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

layer_meta = {}
grid_key = None
for role, path in exported.items():
    with rasterio.open(path) as src:
        data = src.read(1)
        current_grid = (src.width, src.height, str(src.crs), tuple(src.transform))
        if grid_key is None:
            grid_key = current_grid
        elif current_grid != grid_key:
            raise RuntimeError(f'Grid mismatch for {role}: {current_grid} != {grid_key}')
        finite = np.isfinite(data)
        values = data[finite]
        if role == 'vstd' and (not finite.any() or np.any(values < 0)):
            raise RuntimeError('vstd is empty or contains negative values')
        nodata_value = src.nodata
        if isinstance(nodata_value, float) and not np.isfinite(nodata_value):
            nodata_value = None
        layer_meta[role] = {
            'path': str(path), 'sha256': sha256(path), 'bytes': path.stat().st_size,
            'shape': [src.height, src.width], 'crs': str(src.crs),
            'transform': list(src.transform), 'nodata': nodata_value,
            'finite_pixels': int(finite.sum()),
            'min': float(values.min()) if values.size else None,
            'median': float(np.median(values)) if values.size else None,
            'max': float(values.max()) if values.size else None,
        }

manifest = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME, 'orbit': 'ascending', 'start': START, 'end': END,
    'multilook': NLOOK, 'licsbas_commit': actual_commit,
    'steps': ['01', '02', '11', '12', '13', '14'],
    'layers': layer_meta,
}
manifest_path = DRIVE_OUT / 'jakarta_licsbas_2020_2023_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('Manifest:', manifest_path)


In [ ]:
# 13 — Archive final results and verify them
RESULTS_ARCHIVE = ARCHIVE / 'TS_GEOCml10_results_2020_2023.tar.gz'
items = ['TS_GEOCml10/results', 'TS_GEOCml10/info', 'TS_GEOCml10/network', 'GEOCml10/EQA.dem_par']
existing_items = [item for item in items if (WORK / item).exists()]
archive_run = subprocess.run(
    ['tar', '-czf', str(RESULTS_ARCHIVE), '-C', str(WORK), *existing_items]
)
if archive_run.returncode != 0 or not RESULTS_ARCHIVE.exists() or RESULTS_ARCHIVE.stat().st_size == 0:
    raise RuntimeError('Final-results archive failed')
subprocess.run(['tar', '-tzf', str(RESULTS_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
print(f'VERIFIED final archive: {RESULTS_ARCHIVE} ({RESULTS_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('The matched GeoTIFF stack and provenance manifest are ready for local ingestion.')
